In [ ]:
import os
import re
import csv
import cv2
import numpy as np
from google.colab import files

# =============================================================================
# CONFIGURACION
# =============================================================================

BASE_PROYECTO = "/content/proyecto_ia"

DIR_IMAGENES_ENTRENAMIENTO = os.path.join(BASE_PROYECTO, "imagenes_binarias_entrenamiento")
DIR_IMAGENES_PRUEBA        = os.path.join(BASE_PROYECTO, "imagenes_binarias_prueba")
DIR_CSV_ENTRENAMIENTO      = os.path.join(BASE_PROYECTO, "CSV_entrenamiento")
DIR_CSV_PRUEBA             = os.path.join(BASE_PROYECTO, "CSV_prueba")

RUTA_CSV_ENTRENAMIENTO = os.path.join(DIR_CSV_ENTRENAMIENTO, "datos_entrenamiento.csv")
RUTA_CSV_PRUEBA        = os.path.join(DIR_CSV_PRUEBA, "datos_prueba.csv")

ANCHO, ALTO = 128, 128
EXTENSIONES_VALIDAS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

# =============================================================================
# FUNCIONES
# =============================================================================

def crear_estructura_carpetas():
    for ruta in [DIR_IMAGENES_ENTRENAMIENTO, DIR_IMAGENES_PRUEBA,
                 DIR_CSV_ENTRENAMIENTO, DIR_CSV_PRUEBA]:
        if not os.path.exists(ruta):
            os.makedirs(ruta)
            print("Carpeta creada: " + ruta)

def subir_imagenes(destino):
    print("\nSelecciona las imagenes para: " + os.path.basename(destino))
    subidos = files.upload()
    for nombre in subidos.keys():
        os.rename(nombre, os.path.join(destino, nombre))
    print("Se han movido " + str(len(subidos)) + " archivos a " + destino)

def leer_imagen_binaria_como_vector(ruta_imagen):
    imagen = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
    if imagen is None:
        raise ValueError("No se pudo leer: " + ruta_imagen)
    imagen = cv2.resize(imagen, (ANCHO, ALTO), interpolation=cv2.INTER_NEAREST)
    imagen_binaria = np.where(imagen > 127, 1, 0).astype(np.uint8)
    return imagen_binaria.flatten().tolist()

def obtener_imagenes_validas(directorio):
    return [os.path.join(directorio, f) for f in os.listdir(directorio)
            if os.path.isfile(os.path.join(directorio, f))
            and f.lower().endswith(EXTENSIONES_VALIDAS)]

def extraer_numero_entrenamiento(ruta_imagen):
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0]
    coincidencia = re.match(r"^[12]\.(\d+)$", nombre)
    return int(coincidencia.group(1)) if coincidencia else 999999

def extraer_numero_prueba(ruta_imagen):
    nombre = os.path.splitext(os.path.basename(ruta_imagen))[0].lower()
    coincidencia = re.search(r"(\d+)$", nombre)
    return int(coincidencia.group(1)) if coincidencia else 999999

def clasificar_imagen_entrenamiento(ruta_imagen):
    nombre = os.path.basename(ruta_imagen)
    if nombre.startswith("1."): return 0
    if nombre.startswith("2."): return 1
    raise ValueError("Nombre invalido en entrenamiento: " + nombre)

def clasificar_imagen_prueba(ruta_imagen):
    nombre = os.path.basename(ruta_imagen).lower()
    if nombre.startswith(("negativa", "negativo")): return 0
    if nombre.startswith(("positiva", "positivo")): return 1
    raise ValueError("Nombre invalido en prueba: " + nombre)

def generar_csv(directorio_imgs, ruta_csv, func_clasificar, func_extraer, titulo):
    separador = "=" * 40
    print("\n" + separador)
    print(titulo)
    print(separador)
    imgs = obtener_imagenes_validas(directorio_imgs)
    imgs.sort(key=func_extraer)

    if not imgs:
        print("Aviso: No hay imagenes en " + directorio_imgs)
        return

    with open(ruta_csv, mode="w", newline="") as f:
        escritor = csv.writer(f)
        # Encabezado: pixel_1, pixel_2, ..., pixel_16384, etiqueta
        n_pixeles = ANCHO * ALTO
        encabezado = ["pixel_" + str(i + 1) for i in range(n_pixeles)] + ["etiqueta"]
        escritor.writerow(encabezado)

        for i, ruta in enumerate(imgs, 1):
            etiqueta = func_clasificar(ruta)
            vector   = leer_imagen_binaria_como_vector(ruta)
            escritor.writerow(vector + [etiqueta])
            print("Procesado " + str(i).zfill(2) + ": " + os.path.basename(ruta) + " -> Label " + str(etiqueta))

    print("Archivo guardado en : " + ruta_csv)
    print("Total procesadas    : " + str(len(imgs)) + " imagenes")

# =============================================================================
# EJECUCION
# =============================================================================

crear_estructura_carpetas()

# Sube tus imagenes de entrenamiento (positivas: 2.x.png, negativas: 1.x.png)
subir_imagenes(DIR_IMAGENES_ENTRENAMIENTO)

# Sube tus imagenes de prueba (positivas: positiva_x.png, negativas: negativa_x.png)
subir_imagenes(DIR_IMAGENES_PRUEBA)

# Genera los CSVs
generar_csv(DIR_IMAGENES_ENTRENAMIENTO, RUTA_CSV_ENTRENAMIENTO,
            clasificar_imagen_entrenamiento, extraer_numero_entrenamiento, "ENTRENAMIENTO")

generar_csv(DIR_IMAGENES_PRUEBA, RUTA_CSV_PRUEBA,
            clasificar_imagen_prueba, extraer_numero_prueba, "PRUEBA")

print("\nProceso finalizado.")
